# Build your satellite-data inventory. Search it again and again.

Save Sentinel-2 metadata for a small study area around Madrid, then explore different dates and scenes without querying the provider every time. You will also try live fallback, refresh the inventory, and inspect its writer version. **Imagery is not downloaded.** Previews and live catalog queries require internet.

Run cells from top to bottom in Jupyter or a standard Google Colab Python runtime. The example makes a few small public catalog requests. Results can change or be empty; failures are displayed explicitly. Colab files disappear when its runtime ends—download the archive at the end.

This notebook requires SuperSTAC **0.3.x**. Before the release is published, install your locally built v0.3 wheel and skip the installation cell. [Open this notebook in Google Colab](https://colab.research.google.com/drive/1JIx8j0Vi7jypKe5Q6klgCzYQMFHMaKGc?usp=sharing).

In [ ]:
%pip install -q "superstac>=0.3,<0.4" pyarrow

In [ ]:
import gc
import json
import os
from pathlib import Path
import shutil
import tempfile
import time

import pyarrow.parquet as pq
import superstac
from superstac import Client, AsyncClient
from IPython.display import display, FileLink, Image

print("SuperSTAC:", superstac.__version__)
print("GeoParquet available:", superstac.geoparquet_available)
assert superstac.geoparquet_available, "Install a GeoParquet-enabled v0.3 build."
assert hasattr(Client, "ingest"), "Restart the kernel after installing v0.3."


## 1. Choose a small area and date window

A unique temporary directory prevents this demo from overwriting an existing inventory. Keep the default bounds for a quick run. Raising the search limit does not affect ingestion: ingestion follows all pages within its scope.

In [ ]:
CATALOG_ID = "earth-search"
CATALOG_URL = os.environ.get("SUPERSTAC_DEMO_CATALOG_URL", "https://earth-search.aws.element84.com/v1")
COLLECTION = "sentinel-2-l2a"
BBOX = [-3.8, 40.3, -3.6, 40.5]
WEEK_ONE = "2025-02-01T00:00:00Z/2025-02-08T00:00:00Z"
WEEK_TWO = "2025-02-08T00:00:00Z/2025-02-15T00:00:00Z"
DATASET = Path(tempfile.mkdtemp(prefix="superstac-geoparquet-demo-", dir=Path.cwd()))
CATALOGS = [{"id": CATALOG_ID, "url": CATALOG_URL}]
SETTINGS = {"logging_enabled": False, "enable_background_health_monitor": False,
            "search_healthy_catalogs_only": False, "max_retry_attempts": 1,
            "per_catalog_timeout_seconds": 30}
client = Client(catalogs=CATALOGS, settings=SETTINGS)
print("Dataset:", DATASET)


## 2. Ingest with progress

The callback reports received and durably saved metadata records. A provider may omit a total count, so a percentage is not always available. If interrupted, repeat the call with `resume=True` and the same scope, name, and page size.

In [ ]:
last_progress = 0.0

def progress(event):
    global last_progress
    now = time.monotonic()
    if now - last_progress >= 1 or event["phase"] in ("completed", "failed"):
        print(f"{event['phase']}: {event['items_received']} received, "
              f"{event['items_saved']} saved, {event['files']} files")
        last_progress = now

snapshot = client.ingest(
    CATALOG_ID, str(DATASET), name="madrid-week-one",
    collections=[COLLECTION], bbox=BBOX, datetime=WEEK_ONE,
    page_size=100, items_per_file=1000, max_dataset_mib=128,
    progress=progress,
)
assert snapshot["complete"]
print("Saved metadata rows:", snapshot["items"])


## 3. Search and discover locally

This search stays within saved coverage and requires no catalog API request. `sortby` applies before the per-catalog limit. Collection descriptions summarize the saved inventory, not the provider's full collection.

In [ ]:
local = Client(catalogs=CATALOGS, settings=SETTINGS,
               mode="snapshot", dataset=str(DATASET))
query = dict(collections=[COLLECTION], bbox=BBOX, datetime=WEEK_ONE,
             sortby=["-datetime"], limit=5)
started = time.perf_counter()
result = local.search(**query)
print(f"Local search: {(time.perf_counter() - started) * 1000:.1f} ms")
print(json.dumps(result.metadata, indent=2))
assert result.metadata["catalogs_failed"] == 0, result.metadata["failures"]
print("Item IDs:", [item["id"] for item in result.items()])
print("Collections:", local.list_collections())


### Optional online preview

Saved metadata contains asset URLs, not image pixels. Some providers require signing or authentication. If no thumbnail is advertised, this cell simply reports that.

In [ ]:
thumbnail = next((asset["href"] for item in result.items()
                  for key, asset in item.get("assets", {}).items()
                  if key == "thumbnail" or "thumbnail" in asset.get("roles", [])), None)
if thumbnail:
    display(Image(url=thumbnail, width=500))
else:
    print("No thumbnail URL in these results; inspect item assets for available previews.")


## 4. Inspect the file's writer version

The Parquet footer includes both standard GeoParquet metadata and SuperSTAC provenance. The file's writer version can differ from the version currently installed. Empty scopes have no Parquet files.

In [ ]:
manifest = json.loads((DATASET / "manifest.json").read_text())
files = manifest["catalogs"][CATALOG_ID]["madrid-week-one"]["files"]
if files:
    footer = pq.read_metadata(DATASET / files[0]["path"])
    print("created_by:", footer.created_by)
    producer = json.loads(footer.metadata[b"superstac"])
    print("SuperSTAC provenance:", producer)
    assert producer["version"] == superstac.__version__
    assert b"geo" in footer.metadata and b"stac-geoparquet" in footer.metadata
else:
    print("Complete empty inventory: no Parquet files to inspect.")


## 5. Retain a second scope and search their union

Adjacent scopes can jointly cover a larger query; gaps cannot. Reopen snapshot clients to see a newly published manifest. Duplicate collection/item IDs are returned once.

In [ ]:
client.ingest(CATALOG_ID, str(DATASET), name="madrid-week-two",
              collections=[COLLECTION], bbox=BBOX, datetime=WEEK_TWO,
              max_dataset_mib=128, progress=progress)
local.shutdown()
del local
gc.collect()
local = Client(catalogs=CATALOGS, settings=SETTINGS,
               mode="snapshot", dataset=str(DATASET))
union = local.search(collections=[COLLECTION], bbox=BBOX,
                     datetime="2025-02-01T00:00:00Z/2025-02-15T00:00:00Z",
                     sortby=["-datetime"], limit=10)
assert union.metadata["catalogs_failed"] == 0, union.metadata["failures"]
print("Named scopes:", list(json.loads((DATASET / "manifest.json").read_text())["catalogs"][CATALOG_ID]))
print("Union returned:", len(union))


## 6. Automatic API fallback

Auto uses fresh, complete local coverage where available. March is outside these saved scopes, so this query goes to the API. A zero freshness limit also forces live requests. Auto does not save its API results as complete inventories.

In [ ]:
auto = Client(catalogs=CATALOGS, settings=SETTINGS, mode="auto",
              dataset=str(DATASET), max_snapshot_age_seconds=86400)
manifest_before = (DATASET / "manifest.json").read_bytes()
live_result = auto.search(collections=[COLLECTION], bbox=BBOX,
                          datetime="2025-03-01T00:00:00Z/2025-03-08T00:00:00Z", limit=3)
print(json.dumps(live_result.metadata, indent=2))
assert live_result.metadata["catalogs_failed"] == 0, live_result.metadata["failures"]
assert manifest_before == (DATASET / "manifest.json").read_bytes()
auto.shutdown()


## 7. Incremental acquisition-window refresh

Extend week two by a day while re-fetching an overlapping window. New records override older records with the same collection/item ID. This does **not** detect deletions or late changes outside that window. Omit `incremental_since` for a full refresh. Untouched history retains its original freshness timestamp.

In [ ]:
updated = client.ingest(
    CATALOG_ID, str(DATASET), name="madrid-week-two",
    collections=[COLLECTION], bbox=BBOX,
    datetime="2025-02-08T00:00:00Z/2025-02-16T00:00:00Z",
    incremental_since="2025-02-14T00:00:00Z",
    max_dataset_mib=128, progress=progress,
)
print("Overlay present:", updated["overlays"])
assert updated["overlays"]


## 8. Compact and reclaim retired files

Compaction resolves duplicate versions and rewrites bounded files. Old readers remain usable. Applied cleanup refuses active snapshot readers, so release them first; `shutdown()` alone does not release their inventory handle. The calls below only affect the temporary demo dataset.

In [ ]:
client.compact_dataset(str(DATASET), items_per_file=1000, max_dataset_mib=128)
local.shutdown()
del local
gc.collect()
preview = client.cleanup_dataset(str(DATASET))
print("Reclaimable bytes:", preview["bytes"])
print("Retired files:", preview["files"])
cleaned = client.cleanup_dataset(str(DATASET), apply=True)
print("Cleanup applied:", cleaned["applied"])


## 9. Async clients use the same backend

Jupyter and Colab support top-level `await`; there is no need for `asyncio.run()` here. Async ingestion and maintenance use the same keyword options as their synchronous equivalents.

In [ ]:
async_client = AsyncClient(catalogs=CATALOGS, settings=SETTINGS,
                           mode="snapshot", dataset=str(DATASET))
async_result = await async_client.search(**query)
assert async_result.metadata["catalogs_failed"] == 0
print("Async local results:", len(async_result))
await async_client.shutdown()
del async_client
gc.collect()


## 10. Download the inventory and support information

The archive includes the manifest, metadata files, and a small environment report. It contains no imagery. Download it before ending a Colab runtime. When filing a bug, include the package version, a minimal query, and failure metadata; remove credentials or sensitive data from any attachment.

In [ ]:
client.shutdown()
support = {"superstac_version": superstac.__version__,
           "query": query, "metadata": result.metadata}
(DATASET / "demo-report.json").write_text(json.dumps(support, indent=2))
archive = shutil.make_archive(str(DATASET), "zip", root_dir=DATASET)
print("Saved:", archive)
try:
    from google.colab import files as colab_files
except ImportError:
    display(FileLink(Path(archive).name))
else:
    colab_files.download(archive)


## Next steps

- [GeoParquet guide](https://spatialnode.com/superstac/docs/guides/geoparquet)
- [Measured local-scan benchmark](https://spatialnode.com/superstac/docs/guides/benchmarks)
- [What’s new](https://spatialnode.com/superstac/docs/releases)

This notebook's single timing is a demo, not a benchmark. The documented benchmark compares the same scanner with and without pruning, asserts equal results, and reports repeated measurements.